In [1]:
import re
import torch
import pandas as pd
import numpy as np
from transformers import BartTokenizer, BartForSequenceClassification

# =========================
# CONFIG
# =========================
MODEL_DIR = "./bart_relevancia_model"   # pasta do seu modelo treinado
MAX_LENGTH = 256

# limiares
LIMIAR_RELEVANTE = 0.85
LIMIAR_REVISAR = 0.40

# termos fortes por regra
TERMOS_POSITIVOS_FORTES = [
    "controle de acesso",
    "controle de acesso facial",
    "catraca",
    "catracas",
    "leitor facial",
    "leitores faciais",
    "reconhecimento facial",
    "biometria facial",
    "biometria de face",
    "relógio de ponto",
    "controle de ponto",
    "registro de frequência",
    "controle de frequência",
    "registrador eletrônico de ponto",
    "rep",
    "rep-c",
    "rep-p",
    "idface",
    "hid",
    "fechadura biométrica",
    "fechadura eletromagnética",
    "cancela eletrônica",
    "cancela automática",
    "torniquete",
    "botoeira",
    "porta giratória",
    "detector de metais",
    "software gerenciador de controle de acesso",
]

TERMOS_NEGATIVOS_FORTES = [
    "medicamentos",
    "cadeiras escolares",
    "merenda escolar",
    "gêneros alimentícios",
    "material de limpeza",
    "combustível",
    "combustíveis",
    "frota",
    "rastreamento veicular",
    "gps",
    "telemetria",
    "uniforme",
    "livros didáticos",
    "software de multas",
    "prontuário eletrônico",
    "erp hospitalar",
    "buffet",
    "academia ao ar livre",
    "estação solda",
    "solda",
]

# =========================
# CARREGAR MODELO
# =========================
tokenizer = BartTokenizer.from_pretrained(MODEL_DIR)
model = BartForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# =========================
# LIMPEZA DE TEXTO
# =========================
def limpar_texto(texto: str) -> str:
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().lower()
    texto = re.sub(r"\s+", " ", texto)
    return texto

# =========================
# REGRAS EXPLICÁVEIS
# =========================
def classificar_por_regras(texto: str):
    t = limpar_texto(texto)

    positivos_encontrados = [termo for termo in TERMOS_POSITIVOS_FORTES if termo in t]
    negativos_encontrados = [termo for termo in TERMOS_NEGATIVOS_FORTES if termo in t]

    # negativo forte ganha se não houver evidência positiva muito clara
    if negativos_encontrados and not positivos_encontrados:
        return {
            "classe_regra": 0,
            "status_regra": "nao_relevante",
            "motivo_regra": f"Termos negativos fortes encontrados: {', '.join(negativos_encontrados[:5])}"
        }

    # positivo forte
    if positivos_encontrados and not negativos_encontrados:
        return {
            "classe_regra": 1,
            "status_regra": "relevante",
            "motivo_regra": f"Termos positivos fortes encontrados: {', '.join(positivos_encontrados[:5])}"
        }

    # conflito ou ausência de termos
    if positivos_encontrados and negativos_encontrados:
        return {
            "classe_regra": "",
            "status_regra": "revisar",
            "motivo_regra": f"Conflito entre termos positivos ({', '.join(positivos_encontrados[:3])}) e negativos ({', '.join(negativos_encontrados[:3])})"
        }

    return {
        "classe_regra": "",
        "status_regra": "sem_regra",
        "motivo_regra": "Nenhuma regra forte acionada"
    }

# =========================
# MODELO
# =========================
def prever_probabilidades(textos):
    inputs = tokenizer(
        textos,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    return probs

# =========================
# DECISÃO FINAL
# =========================
def decidir_final(prob_relevante: float, resultado_regra: dict):
    classe_regra = resultado_regra["classe_regra"]
    status_regra = resultado_regra["status_regra"]

    # 1) Se regra forte decidiu algo, ela pode sobrescrever
    if status_regra == "relevante":
        return 1, "relevante", "regra_forte_positiva"

    if status_regra == "nao_relevante":
        return 0, "nao_relevante", "regra_forte_negativa"

    # 2) Senão, usa confiança do modelo
    if prob_relevante >= LIMIAR_RELEVANTE:
        return 1, "relevante", "modelo_confiante"

    if prob_relevante >= LIMIAR_REVISAR:
        return "", "revisar", "modelo_inseguro"

    return 0, "nao_relevante", "modelo_confiante"

# =========================
# CLASSIFICAÇÃO EM LOTE
# =========================
def classificar_textos(textos):
    textos_limpos = [limpar_texto(t) for t in textos]
    probs = prever_probabilidades(textos_limpos)

    resultados = []
    for texto_original, texto_limpo, prob in zip(textos, textos_limpos, probs):
        prob_nao_relevante = float(prob[0])
        prob_relevante = float(prob[1])

        regra = classificar_por_regras(texto_limpo)
        classificacao_final, status_final, origem_decisao = decidir_final(prob_relevante, regra)

        resultados.append({
            "texto": texto_original,
            "texto_limpo": texto_limpo,
            "classificacao_final": classificacao_final,
            "status_final": status_final,
            "origem_decisao": origem_decisao,
            "classe_regra": regra["classe_regra"],
            "status_regra": regra["status_regra"],
            "motivo_regra": regra["motivo_regra"],
            "prob_nao_relevante": round(prob_nao_relevante, 6),
            "prob_relevante": round(prob_relevante, 6),
        })

    return pd.DataFrame(resultados)

# =========================
# EXEMPLO COM LISTA
# =========================
if __name__ == "__main__":
    textos_teste = [
        "Estação Solda tipo corrente: alternada, tensão alimentação: 110, tipo ponta: removível",
        "LOCAÇÃO DE CATRACA ELETRÔNICA TIPO PEDESTAL COM SOFTWARE GERENCIADOR DE CONTROLE DE ACESSO",
        "Aquisição de medicamentos hospitalares",
        "Terminal de controle de acesso multibiométrico com reconhecimento facial",
        "Software de gestão de multas e infrações de trânsito",
        "Relógio de ponto biométrico com emissão de AFD",
        "Serviço de vigilância patrimonial com controle de entrada e saída de pessoas",
    ]

    df_resultado = classificar_textos(textos_teste)
    print(df_resultado)

    # salvar
    df_resultado.to_csv("saida_classificada.csv", index=False, encoding="utf-8-sig")
    print("Arquivo salvo: saida_classificada.csv")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/263 [00:00<?, ?it/s]

                                               texto  \
0  Estação Solda tipo corrente: alternada, tensão...   
1  LOCAÇÃO DE CATRACA ELETRÔNICA TIPO PEDESTAL CO...   
2             Aquisição de medicamentos hospitalares   
3  Terminal de controle de acesso multibiométrico...   
4  Software de gestão de multas e infrações de tr...   
5     Relógio de ponto biométrico com emissão de AFD   
6  Serviço de vigilância patrimonial com controle...   

                                         texto_limpo  classificacao_final  \
0  estação solda tipo corrente: alternada, tensão...                    0   
1  locação de catraca eletrônica tipo pedestal co...                    1   
2             aquisição de medicamentos hospitalares                    0   
3  terminal de controle de acesso multibiométrico...                    1   
4  software de gestão de multas e infrações de tr...                    0   
5     relógio de ponto biométrico com emissão de afd                    1   
6  serviço d

### Se quiser ler de um CSV com coluna texto, use este trecho:

In [ ]:
import pandas as pd

df_entrada = pd.read_csv("entradas.csv")
df_saida = classificar_textos(df_entrada["texto"].astype(str).tolist())
df_saida.to_csv("saida_classificada.csv", index=False, encoding="utf-8-sig")
print("Arquivo salvo: saida_classificada.csv")